In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix
import time
import numpy as np
import pandas as pd

In [2]:
device =  torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"Using Device: {device}")

In [3]:
# Set and configure classification models
num_classes = 26

# VGG16 set-up
vgg16 = models.vgg16(weights=None)

vgg16.classifier = nn.Sequential(
    nn.Linear(25088, 512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512, num_classes)
)

vgg16.load_state_dict(
    torch.load(
        "models/vgg16_best.pth",
        map_location=device
    )
)

vgg16 = vgg16.to(device)
vgg16.eval()

# RESNET50 set-up

resnet50 = models.resnet50(weights=None)

resnet50.fc = nn.Sequential(
    nn.Linear(resnet50.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

resnet50.load_state_dict(
    torch.load(
        "models/resnet50_best.pth",
        map_location=device
    )
)

resnet50 = resnet50.to(device)

resnet50.eval()

# MOBILENETV2 set-up
mobilenetv2 = models.mobilenet_v2(weights=None)

mobilenetv2.classifier = nn.Sequential(
    nn.Linear(mobilenetv2.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

mobilenetv2.load_state_dict(
    torch.load(
        "models/mobilenetv2_best.pth",
        map_location=device
    )
)

mobilenetv2 = mobilenetv2.to(device)
mobilenetv2.eval()

# EFFICIENTNETB0 set-up

efficientnetb0 = models.efficientnet_b0(weights=None)

# replace classifier
efficientnetb0.classifier = nn.Sequential(
    nn.Linear(efficientnetb0.classifier[1].in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, num_classes)
)

# load trained weights
efficientnetb0.load_state_dict(
    torch.load(
        "models/efficientnetb0_best.pth",
        map_location=device
    )
)

efficientnetb0 = efficientnetb0.to(device)

efficientnetb0.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [4]:
# Set paths and loaders
# set paths
TEST_PATH = "smartvision_dataset/classification/test"
TRAIN_PATH = "smartvision_dataset/classification/train"

# create transform
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# set train dataset
train_dataset = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)

# set test_dataset and test_loader
test_dataset  = datasets.ImageFolder(TEST_PATH, transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
CLASS_NAMES = train_dataset.classes

In [6]:
# Create evaluation function
def eval_model(model, test_loader, class_names, device):

    model.eval()

    all_preds = []
    all_labels = []

    correct = 0
    total = 0

    inference_times = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            start_time = time.time()

            outputs = model(images)

            end_time = time.time()

            inference_times.append(end_time - start_time)

            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total

    avg_inference_time = np.mean(inference_times)

    report = classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds)

    return accuracy, avg_inference_time, report, cm

In [7]:
# Calculate Metrics for each Model
# 1. vgg16
vgg_acc, vgg_time, vgg_report, vgg_cm = eval_model(
    vgg16,
    test_loader,
    CLASS_NAMES,
    device
)
# 2. resnet50
resnet_acc, resnet_time, resnet_report, resnet_cm = eval_model(
    resnet50,
    test_loader,
    CLASS_NAMES,
    device
)
# 3. mobilenet
mobilenet_acc, mobilenet_time, mobilenet_report, mobilenet_cm = eval_model(
    mobilenetv2,
    test_loader,
    CLASS_NAMES,
    device
)
# 4. efficientnet
efficientnet_acc, efficientnet_time, efficientnet_report, efficientnet_cm = eval_model(
    efficientnetb0,
    test_loader,
    CLASS_NAMES,
    device
)

In [8]:
# Save Model Results 
model_results = {

    "VGG16": {
        "accuracy": vgg_acc,
        "precision": vgg_report["weighted avg"]["precision"],
        "recall": vgg_report["weighted avg"]["recall"],
        "f1": vgg_report["weighted avg"]["f1-score"],
        "inference_time": vgg_time,
        "model_size": os.path.getsize("models/vgg16_best.pth") / (1024 * 1024)
    },

    "ResNet50": {
        "accuracy": resnet_acc,
        "precision": resnet_report["weighted avg"]["precision"],
        "recall": resnet_report["weighted avg"]["recall"],
        "f1": resnet_report["weighted avg"]["f1-score"],
        "inference_time": resnet_time,
        "model_size": os.path.getsize("models/resnet50_best.pth") / (1024 * 1024)
    },

    "MobileNetV2": {
        "accuracy": mobilenet_acc,
        "precision": mobilenet_report["weighted avg"]["precision"],
        "recall": mobilenet_report["weighted avg"]["recall"],
        "f1": mobilenet_report["weighted avg"]["f1-score"],
        "inference_time": mobilenet_time,
        "model_size": os.path.getsize("models/mobilenetv2_best.pth") / (1024 * 1024)
    },

    "EfficientNetB2": {
        "accuracy": efficientnet_acc,
        "precision": efficientnet_report["weighted avg"]["precision"],
        "recall": efficientnet_report["weighted avg"]["recall"],
        "f1": efficientnet_report["weighted avg"]["f1-score"],
        "inference_time": efficientnet_time,
        "model_size": os.path.getsize("models/efficientnetb0_best.pth") / (1024 * 1024)
    }
}

In [15]:
# save result 
df = pd.DataFrame(model_results).T.reset_index()

df

,index,accuracy,precision,recall,f1,inference_time,model_size
0,VGG16,0.528205,0.563262,0.528205,0.512188,4.835642,105.195729
1,ResNet50,0.505128,0.509741,0.505128,0.481047,1.981689,92.008281
2,MobileNetV2,0.356410,0.377326,0.356410,0.327234,0.526402,9.996606
3,EfficientNetB2,0.469231,0.426508,0.469231,0.422379,0.604541,16.858975


In [16]:
df.to_csv("model_results.csv", index=False)

In [ ]:
# create confusion matrix dict
cm_dict = {
    "VGG16": vgg_cm,
    "ResNet50": resnet_cm,
    "MobileNetV2": mobilenet_cm,
    "EfficientNetB0": efficientnet_cm
}

In [19]:
# save as pickle
import pickle
with open("cm_dict.pkl", "wb") as f:
    pickle.dump(cm_dict, f)